# Anime Recommendation System - Hybrid Collaborative Filtering DAE

This notebook builds a **Hybrid Multi-Task Denoising Autoencoder (DAE)**. It learns from implicit feedback (presence), explicit feedback (ratings), **and** side-information (Content Profile) to handle the cold start problem and improve recommendations.

**Dataset**: [Anime Dataset (Jan 1917 to Oct 2025)](https://www.kaggle.com/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025)
**Framework**: JAX + Flax

Based on the v3 DAE model, this v4 model features:
- Dual-head decoder (Presence prediction + Rating regression)
- Uncertainty weighting for multi-task loss balance
- **Item Features Injection**: We extract categorical tags and semantic synopsis embeddings from `details.csv` and inject a User Content Profile into the DAE's encoder layer.

In [ ]:
import os
import ast
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
from sklearn.preprocessing import MultiLabelBinarizer

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# config
CORPUS_SIZE = 10000
MIN_USER_RATINGS = 10
BATCH_SIZE = 512
HIDDEN_DIM = 4096
BOTTLENECK_DIM = 1024
LEARNING_RATE = 2e-4
DROPOUT_RATE = 0.5
EPOCHS = 30

DATA_DIR = '/kaggle/input/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025/datasets'
RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.csv')
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')

## 1. Load Ratings Data

In [ ]:
print("Loading ratings datasets...")
ratings_df = pd.read_csv(RATINGS_PATH, usecols=['username', 'anime_id', 'score', 'status'])

# Determine top anime for the corpus
anime_counts = ratings_df['anime_id'].value_counts()
corpus_ids = anime_counts.head(CORPUS_SIZE).index.values
corpus_id_to_idx = {aid: idx for idx, aid in enumerate(corpus_ids)}

# Filter ratings to only include corpus items
ratings_df = ratings_df[ratings_df['anime_id'].isin(corpus_id_to_idx)]

# Filter users with less than MIN_USER_RATINGS
user_counts = ratings_df['username'].value_counts()
valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings_df = ratings_df[ratings_df['username'].isin(valid_users)]

print(f"Retained {len(valid_users)} users and {len(ratings_df)} ratings.")

# Save the corpus mapping for backend use
import json
out_dir = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(out_dir, exist_ok=True)
corpus_path = os.path.join(out_dir, "corpus_mapping.json")
with open(corpus_path, 'w') as f:
    json.dump({'corpus_ids': corpus_ids.tolist()}, f)
print(f"Saved corpus mapping to {corpus_path}")

## 2. Load Item Hybrid Features

In [ ]:
print("Loading details dataset for Hybrid Features...")
details_df = pd.read_csv(DETAILS_PATH)

def parse_list(x):
    if pd.isna(x): return []
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            try: return ast.literal_eval(x)
            except: pass
        return [i.strip() for i in x.split(',') if i.strip()]
    return []

# Create aligned corpus dataframe to maintain exact size and ordering
corpus_df = pd.DataFrame({'mal_id': corpus_ids, 'idx': range(CORPUS_SIZE)})
corpus_details = pd.merge(corpus_df, details_df, on='mal_id', how='left')

corpus_details['genres'] = corpus_details['genres'].fillna("[]").apply(parse_list)
corpus_details['themes'] = corpus_details['themes'].fillna("[]").apply(parse_list)
corpus_details['demographics'] = corpus_details['demographics'].fillna("[]").apply(parse_list)
corpus_details['synopsis'] = corpus_details['synopsis'].fillna("")

def combine_tags(row):
    tags = set()
    tags.update(row['genres'])
    tags.update(row['themes'])
    tags.update(row['demographics'])
    return list(tags)

corpus_details['meta_tags'] = corpus_details.apply(combine_tags, axis=1)

mlb = MultiLabelBinarizer()
categorical_matrix = mlb.fit_transform(corpus_details['meta_tags']).astype(np.float32)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    try:
        torch.ones(1, device="cuda")
    except Exception as e:
        print(f"CUDA is available but incompatible ({e}). Falling back to CPU.")
        device = "cpu"

print(f"Loading Sentence Transformer on {device.upper()}...")
model_st = SentenceTransformer('all-MiniLM-L6-v2', device=device)
semantic_matrix = model_st.encode(corpus_details['synopsis'].tolist(), show_progress_bar=True, batch_size=128)

# Zero-out embeddings for missing synopses
empty_synopsis_mask = (corpus_details['synopsis'] == "").values
semantic_matrix = (semantic_matrix - semantic_matrix.mean(axis=0)) / (semantic_matrix.std(axis=0) + 1e-6)
semantic_matrix[empty_synopsis_mask] = 0.0

ITEM_FEATURES = np.concatenate([categorical_matrix, semantic_matrix], axis=1)
ITEM_FEAT_DIM = ITEM_FEATURES.shape[1]
print(f"Extracted Item Features Shape: {ITEM_FEATURES.shape}")

## 3. Rating Normalization

In [ ]:
print("Starting vectorized rating normalization...")

ratings_df['idx'] = ratings_df['anime_id'].map(corpus_id_to_idx)

user_stats = ratings_df[ratings_df['score'] > 0].groupby('username')['score'].agg(['mean', 'std']).rename(columns={'mean': 'user_mean', 'std': 'user_std'})
ratings_df = ratings_df.join(user_stats, on='username')
ratings_df['user_mean'] = ratings_df['user_mean'].fillna(5.0)
ratings_df['user_std'] = ratings_df['user_std'].fillna(2.0)

ratings_df['processed_score'] = ratings_df['score'].astype(np.float32)
mask_dropped_unrated = (ratings_df['status'] == 'dropped') & (ratings_df['score'] == 0)
ratings_df.loc[mask_dropped_unrated, 'processed_score'] = -2.0

mask_watched_unrated = (ratings_df['score'] == 0) & (ratings_df['status'] != 'dropped')
ratings_df.loc[mask_watched_unrated, 'processed_score'] = ratings_df.loc[mask_watched_unrated, 'user_mean']

ratings_df.loc[mask_dropped_unrated, 'processed_score'] = ratings_df.loc[mask_dropped_unrated, 'user_mean'] - 1.5 * ratings_df.loc[mask_dropped_unrated, 'user_std']

mu = ratings_df['user_mean']
sigma = ratings_df['user_std'] + 1e-6
z_score = np.clip((ratings_df['processed_score'] - mu) / sigma, -3.0, 3.0)
abs_score = np.clip((ratings_df['processed_score'] - 5.5) / 2.5, -2.5, 2.0)
alpha = np.clip(sigma / 2.6, 0.3, 0.8)
ratings_df['norm_score'] = np.clip(alpha * z_score + (1.0 - alpha) * abs_score, -2.5, 2.5)

ratings_df['user_idx'] = ratings_df['username'].astype('category').cat.codes
num_users = ratings_df['user_idx'].nunique()

user_idxs = ratings_df['user_idx'].values
item_idxs = ratings_df['idx'].values
norm_scores = ratings_df['norm_score'].values
is_rated = ((ratings_df['score'] > 0) | (ratings_df['status'] == 'dropped')).values.astype(np.float32)

col_presence = item_idxs
data_presence = np.ones_like(item_idxs, dtype=np.float32)

col_ratings = item_idxs + CORPUS_SIZE
data_ratings = norm_scores

cols = np.concatenate([col_presence, col_ratings])
rows = np.concatenate([user_idxs, user_idxs])
data = np.concatenate([data_presence, data_ratings])

X_train_sparse = csr_matrix((data, (rows, cols)), shape=(num_users, CORPUS_SIZE * 2), dtype=np.float32)
M_train_sparse = csr_matrix((is_rated, (user_idxs, item_idxs)), shape=(num_users, CORPUS_SIZE), dtype=np.float32)

print(f"Completed! Training data shape: {X_train_sparse.shape}")

## 4. Model Architecture (Hybrid JAX/Flax DAE)

In [ ]:
class HybridRecommender(nn.Module):
    hidden_dim: int = HIDDEN_DIM
    bottleneck_dim: int = BOTTLENECK_DIM
    corpus_size: int = CORPUS_SIZE

    @nn.compact
    def __call__(self, x, item_features, training: bool = False):
        presence = x[:, :self.corpus_size]
        
        # Compute User Content Profile (Hybrid Injection)
        user_content = jnp.dot(presence, item_features)
        counts = jnp.maximum(jnp.sum(presence, axis=1, keepdims=True), 1.0)
        user_content = user_content / counts
        
        # Concatenate CF input with Content Profile
        h_in = jnp.concatenate([x, user_content], axis=1)
        
        # --- Encoder ---
        h = nn.Dense(self.hidden_dim)(h_in)
        h = nn.swish(h)
        bottleneck = nn.Dense(self.bottleneck_dim, name="bottleneck")(h)
        
        if training:
            rng = self.make_rng('noise')
            noise = random.normal(rng, bottleneck.shape) * 0.1
            z = bottleneck + noise
        else:
            z = bottleneck
            
        # --- Decoder Head 1: Item Presence (Logits) ---
        d1 = nn.Dense(self.hidden_dim // 2)(z)
        d1 = nn.swish(d1)
        d1 = nn.Dense(self.hidden_dim)(d1)
        d1 = nn.swish(d1)
        item_logits = nn.Dense(self.corpus_size, name="item_logits")(d1)
        
        # --- Decoder Head 2: Rating Prediction ---
        d2 = nn.Dense(self.hidden_dim // 2)(z)
        d2 = nn.swish(d2)
        d2 = nn.Dense(self.hidden_dim)(d2)
        d2 = nn.swish(d2)
        rating_pred = nn.Dense(self.corpus_size, name="rating_pred")(d2)
        
        log_var_presence = self.param('log_var_presence', nn.initializers.zeros, (1,))
        log_var_rating = self.param('log_var_rating', nn.initializers.zeros, (1,))
        
        return item_logits, rating_pred, log_var_presence, log_var_rating

## 5. Training Loop

In [ ]:
class TrainState(train_state.TrainState):
    key: jax.Array

def create_train_state(rng, learning_rate, item_feat_dim):
    model = HybridRecommender()
    dummy_input = jnp.ones((1, CORPUS_SIZE * 2))
    dummy_feats = jnp.ones((CORPUS_SIZE, item_feat_dim))
    params = model.init({"params": rng, "noise": rng}, dummy_input, dummy_feats)["params"]
    tx = optax.adam(learning_rate)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx, key=rng)

@jax.jit
def train_step(state, batch, rated_mask, item_features):
    presence = batch[:, :CORPUS_SIZE]
    ratings_z = batch[:, CORPUS_SIZE:]
    
    dropout_rng, vae_rng = random.split(state.key)
    
    keep = random.bernoulli(dropout_rng, p=(1.0 - DROPOUT_RATE), shape=presence.shape)
    corrupted_presence = presence * keep
    corrupted_ratings = ratings_z * keep
    x_in = jnp.concatenate([corrupted_presence, corrupted_ratings], axis=1)
    
    def loss_fn(params):
        item_logits, rating_pred, log_var_presence, log_var_rating = state.apply_fn(
            {"params": params}, x_in, item_features, training=True, rngs={"noise": vae_rng}
        )
        
        log_probs = jax.nn.log_softmax(item_logits, axis=1)
        per_user_counts = jnp.maximum(jnp.sum(presence, axis=1), 1.0)
        presence_loss = jnp.mean(-jnp.sum(presence * log_probs, axis=1) / per_user_counts)
        
        err = rating_pred - ratings_z
        per_entry_loss = optax.huber_loss(err, delta=1.0)
        denom_r = jnp.maximum(jnp.sum(rated_mask, axis=1), 1.0)
        rating_loss = jnp.mean(jnp.sum(rated_mask * per_entry_loss, axis=1) / denom_r)
        
        precision_p = jnp.exp(-log_var_presence)
        precision_r = jnp.exp(-log_var_rating)
        weighted_loss = (precision_p * presence_loss + log_var_presence) + \
                        (precision_r * rating_loss + log_var_rating)
        
        return jnp.mean(weighted_loss), (presence_loss, rating_loss)
        
    (loss, (recon_p, recon_r)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(key=dropout_rng)
    return state, loss, recon_p, recon_r

def data_generator(X_sparse, M_sparse, batch_size):
    num_samples = X_sparse.shape[0]
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    for i in range(0, num_samples, batch_size):
        batch_idx = indices[i:i+batch_size]
        yield X_sparse[batch_idx].toarray(), M_sparse[batch_idx].toarray()

rng = random.PRNGKey(42)
state = create_train_state(rng, LEARNING_RATE, ITEM_FEAT_DIM)

# Must pass the item features as a jax array
jnp_item_features = jnp.array(ITEM_FEATURES)

print("Starting Training...")
for epoch in range(EPOCHS):
    gen = data_generator(X_train_sparse, M_train_sparse, BATCH_SIZE)
    epoch_losses = []
    num_batches = int(np.ceil(X_train_sparse.shape[0] / BATCH_SIZE))
    
    for batch_x, batch_m in tqdm(gen, total=num_batches, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        state, loss, p_loss, r_loss = train_step(state, jnp.array(batch_x), jnp.array(batch_m), jnp_item_features)
        epoch_losses.append(loss)
        
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {np.mean(epoch_losses):.4f}")

from flax import serialization
out_dir = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(out_dir, exist_ok=True)
weights_path = os.path.join(out_dir, "model_weights_v4.msgpack")
with open(weights_path, 'wb') as f:
    f.write(serialization.to_bytes(state.params))
print(f"Saved model weights to {weights_path}")

## 6. Generating Recommendations

In [ ]:
def get_recommendations(state, user_idx, top_k=20, logit_weight=0.3):
    user_input = X_train_sparse[user_idx:user_idx+1].toarray()
    item_logits, rating_pred, _, _ = state.apply_fn(
        {"params": state.params}, jnp.array(user_input), jnp_item_features, training=False
    )
    
    logits_1d = item_logits[0]
    preds_1d = rating_pred[0]
    presence_mask = user_input[0, :CORPUS_SIZE]
    
    norm_logits = (logits_1d - jnp.mean(logits_1d)) / (jnp.std(logits_1d) + 1e-6)
    norm_ratings = (preds_1d - jnp.mean(preds_1d)) / (jnp.std(preds_1d) + 1e-6)
    
    combined_score = (logit_weight * norm_logits) + ((1.0 - logit_weight) * norm_ratings)
    masked_score = jnp.where(presence_mask > 0, -jnp.inf, combined_score)
    
    top_indices = jnp.argsort(masked_score)[-top_k:][::-1]
    top_scores = masked_score[top_indices]
    
    anime_ids = [corpus_ids[i] for i in top_indices]
    return anime_ids, top_scores

print("Sample Recommendations for User 0:")
recs, scores = get_recommendations(state, user_idx=10)
for rank, (aid, score) in enumerate(zip(recs, scores), 1):
    print(f"{rank}. Anime ID: {aid} (Score: {score:.3f})")